# Bid-Ask Spread Estimators on Crypto OHLC Data

This notebook resamples the hourly parquet data for `BTCUSDT`, `ETHUSDT`, `DOGEUSDT`, and `SOLUSDT` to daily OHLC bars, computes the Roll (1984), Corwin-Schultz (2012), and Abdi-Ranaldo (2017) spread estimators, and exports a combined figure and CSV to `output/`.


In [1]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pio.templates.default = "plotly_white"

try:
    import kaleido  # noqa: F401
except ImportError as exc:
    raise ImportError("Install kaleido to export Plotly PNG files: pip install kaleido") from exc

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

ASSETS = ["BTCUSDT", "ETHUSDT", "DOGEUSDT", "SOLUSDT"]
WINDOW = 21
ESTIMATOR_COLS = ["spread_roll", "spread_cs", "spread_ar"]
COLORS = {
    "price": "#1f2937",
    "spread_roll": "#2563eb",
    "spread_cs": "#f97316",
    "spread_ar": "#16a34a",
}


In [2]:
def load_daily_ohlc(symbol: str) -> pd.DataFrame:
    frame = pd.read_parquet(DATA_DIR / f"{symbol}_1h.parquet").sort_index()
    frame.index = pd.to_datetime(frame.index, utc=True)
    daily = frame.resample("1D").agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
    })
    return daily.dropna().rename_axis("date")


def roll_spread(close: pd.Series, window: int = WINDOW) -> pd.Series:
    log_returns = np.log(close).diff()
    cov = log_returns.rolling(window=window, min_periods=window).cov(log_returns.shift(1))
    spread = 2 * np.sqrt((-cov).clip(lower=0))
    spread = spread.mask(cov >= 0)
    return spread.rename("spread_roll")


def corwin_schultz_spread(high: pd.Series, low: pd.Series) -> pd.Series:
    constant = 3 - 2 * np.sqrt(2)
    log_range = np.log(high / low)
    beta = log_range.pow(2) + log_range.shift(-1).pow(2)
    high_2d = pd.concat([high, high.shift(-1)], axis=1).max(axis=1)
    low_2d = pd.concat([low, low.shift(-1)], axis=1).min(axis=1)
    gamma = np.log(high_2d / low_2d).pow(2)
    alpha = (np.sqrt(2 * beta) - np.sqrt(beta)) / constant - np.sqrt(gamma / constant)
    spread = 2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha))
    spread = spread.mask(alpha < 0)
    return spread.rename("spread_cs")


def abdi_ranaldo_spread(close: pd.Series, high: pd.Series, low: pd.Series, window: int = WINDOW) -> pd.Series:
    # Abdi-Ranaldo is a relative spread estimator: compute in log-price space.
    log_close = np.log(close)
    log_eta = (np.log(high) + np.log(low)) / 2
    raw = np.sqrt(((log_close - log_eta) * (log_close - log_eta.shift(-1))).clip(lower=0))
    spread = 2 * raw.rolling(window=window, min_periods=window).mean()
    return spread.rename("spread_ar")


def build_spread_frame(symbol: str) -> pd.DataFrame:
    bars = load_daily_ohlc(symbol)
    spreads = pd.concat([
        roll_spread(bars["close"]),
        corwin_schultz_spread(bars["high"], bars["low"]),
        abdi_ranaldo_spread(bars["close"], bars["high"], bars["low"]),
    ], axis=1)
    result = pd.concat([bars[["close"]], spreads], axis=1)
    result["symbol"] = symbol
    return result.reset_index()


spread_df = pd.concat([build_spread_frame(symbol) for symbol in ASSETS], ignore_index=True)
spread_df["date"] = pd.to_datetime(spread_df["date"], utc=True)
spread_df = spread_df[["date", "symbol", "close", *ESTIMATOR_COLS]].sort_values(["symbol", "date"])
spread_df.to_csv(OUTPUT_DIR / "spread_estimates.csv", index=False)
spread_df.head()


FileNotFoundError: [Errno 2] No such file or directory: '/Users/peterprendergast/Library/Mobile Documents/com~apple~CloudDocs/MSCF/COMP0051_AT/coursework/final_project/Algo-Trading-Project/data/BTCUSDT_1h.parquet'

In [ ]:
subplot_titles = []
for symbol in ASSETS:
    subplot_titles.extend([
        f"{symbol} Close Price",
        f"{symbol} Spread Estimates",
        f"{symbol} 21-Day Rolling Means",
        f"{symbol} Estimator Correlations",
    ])

fig = make_subplots(
    rows=len(ASSETS),
    cols=4,
    specs=[[{}, {}, {}, {"type": "heatmap"}] for _ in ASSETS],
    subplot_titles=subplot_titles,
    horizontal_spacing=0.07,
    vertical_spacing=0.06,
)

for row, symbol in enumerate(ASSETS, start=1):
    asset = spread_df.loc[spread_df["symbol"] == symbol].copy()
    asset["date"] = pd.to_datetime(asset["date"], utc=True)
    rolling_means = asset[ESTIMATOR_COLS].rolling(WINDOW, min_periods=WINDOW).mean()

    fig.add_trace(
        go.Scatter(
            x=asset["date"],
            y=asset["close"],
            mode="lines",
            name=f"{symbol} Close",
            line={"color": COLORS["price"], "width": 1.5},
            showlegend=False,
        ),
        row=row,
        col=1,
    )

    for col_name, label in zip(ESTIMATOR_COLS, ["Roll", "Corwin-Schultz", "Abdi-Ranaldo"]):
        showlegend = row == 1
        fig.add_trace(
            go.Scatter(
                x=asset["date"],
                y=asset[col_name],
                mode="lines",
                name=label,
                legendgroup=label,
                line={"color": COLORS[col_name], "width": 1.5},
                showlegend=showlegend,
            ),
            row=row,
            col=2,
        )
        fig.add_trace(
            go.Scatter(
                x=asset["date"],
                y=rolling_means[col_name],
                mode="lines",
                name=f"{label} Mean",
                legendgroup=f"{label} Mean",
                line={"color": COLORS[col_name], "width": 1.8},
                showlegend=False,
            ),
            row=row,
            col=3,
        )

    corr = asset[ESTIMATOR_COLS].corr()
    fig.add_trace(
        go.Heatmap(
            z=corr.values,
            x=["Roll", "Corwin-Schultz", "Abdi-Ranaldo"],
            y=["Roll", "Corwin-Schultz", "Abdi-Ranaldo"],
            zmin=-1,
            zmax=1,
            coloraxis="coloraxis",
            text=np.round(corr.values, 2),
            texttemplate="%{text}",
            hovertemplate="x=%{x}<br>y=%{y}<br>corr=%{z:.2f}<extra></extra>",
            showscale=False,
        ),
        row=row,
        col=4,
    )

    fig.update_xaxes(title_text="Date", row=row, col=1)
    fig.update_xaxes(title_text="Date", row=row, col=2)
    fig.update_xaxes(title_text="Date", row=row, col=3)
    fig.update_xaxes(title_text="Estimator", row=row, col=4)
    fig.update_yaxes(title_text="Close", row=row, col=1)
    fig.update_yaxes(title_text="Spread", row=row, col=2)
    fig.update_yaxes(title_text="21-Day Mean", row=row, col=3)
    fig.update_yaxes(title_text="Estimator", row=row, col=4)

fig.update_layout(
    title="Bid-Ask Spread Estimator Comparison Across Four Crypto Assets",
    height=1400,
    width=1900,
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.01, "xanchor": "left", "x": 0},
    margin={"l": 60, "r": 30, "t": 110, "b": 50},
    coloraxis={"colorscale": "RdBu", "cmid": 0, "colorbar": {"title": "Correlation"}},
)
fig.write_image(OUTPUT_DIR / "spread_comparison.png", scale=2)
fig.show()
